### Example: The promise-version Hopcroft algorithm for DFA minimization in the promise-problem setting
This code implements the promise-version Hopcroft algorithm as described in
my thesis (TODO: add reference). It extends the standard Hopcroft procedure
from the seminal book "Introduction to Automata Theory, Languages, and Computation"
by Hopcroft, Motwani, and Ullman, which identifies the smallest DFA recognizing a given language.

In the promise-problem setting, we are given two disjoint languages L_yes and L_no,
and the task is to find a DFA that accepts all words in L_yes and rejects all words in L_no.
Unlike the standard recognition problem, L_yes ∪ L_no does not need to cover the entire input space.
Consequently, DFA minimization in the promise setting seeks the smallest DFA that may classify
words outside L_yes and L_no arbitrarily, while still correctly classifying all words in L_yes and L_no.
This added freedom can yield smaller DFAs than in the standard recognition problem.

As in the standard Hopcroft algorithm, we iteratively populate a table of all pairs of states
of a candidate DFA via identifying clusters of pairs that can be mapped to each other by
applying the same input sequence to both states in a pair.
From the standard automata theory, two states p and q can be merged if, for all input symbols a,
the states delta(p, a) and delta(q, a) are indistinguishable; i.e., there is no a for which
one is accepting and the other is rejecting.

Thus, we can merge a pair of states if, for all pairs within a cluster, the accept/reject labeling
ensures label(delta(p, a)) == label(delta(q, a)), while enforcing label(p) != label(q) for any (p, q) where p is accepting and
q is rejecting in the candidate DFA (i.e., the initial state and the reject language defining state).

The code below first implements the table-filling procedure to identify clusters of state pairs.
Then, it formulates the merging constraints for pairs within a cluster as a SAT problem and solves
them using the Z3 SMT solver.

In [1]:
from promhop import populate_pair_classes, check_pair_classification_complete, get_keys_from_value
from aalpy.automata import Dfa, DfaState 

In [2]:
# Simple example from (TODO: add the correct reference) where we know from modular arithmetics that
# the DFA-induced promise problem from the 6-cyclic DFA can be reduced to a 2-state DFA.
# Can our promise version Hopcroft algorithm find this reduction?

# First, we define the 6-cyclic DFA in the AALpy framework. 

num_states = 6
states = [DfaState(state_id=f'{i}') for i in range(num_states)]
input_alphabet = ['s']

# only the initial state is accepting
states[0].is_accepting=True

# define transitions to form a 6-cycle
states[0].transitions['s'] = states[1]
states[1].transitions['s'] = states[2]
states[2].transitions['s'] = states[3]
states[3].transitions['s'] = states[4]
states[4].transitions['s'] = states[5]
states[5].transitions['s'] = states[0]

cyclic6 = Dfa(states[0], states)

In [3]:
cyclic_pair_classes = {}

seed_pair = (0,1)
populate_pair_classes(seed_pair, cyclic_pair_classes, cyclic6)

seed_pair = (0,2)
populate_pair_classes(seed_pair, cyclic_pair_classes, cyclic6)

seed_pair = (0,3)
populate_pair_classes(seed_pair, cyclic_pair_classes, cyclic6)

{(0, 1): 0,
 (1, 2): 0,
 (2, 3): 0,
 (3, 4): 0,
 (4, 5): 0,
 (0, 5): 0,
 (0, 2): 1,
 (1, 3): 1,
 (2, 4): 1,
 (3, 5): 1,
 (0, 4): 1,
 (1, 5): 1,
 (0, 3): 2,
 (1, 4): 2,
 (2, 5): 2}

In [4]:
check_pair_classification_complete(cyclic_pair_classes, num_states)

(True, [])

It looks like we’ve classified all pairs into mutually mappable clusters!

Next, we examine whether two states can be merged without substantially altering the DFA’s language. In the standard recognition setting, where every state is accepting or rejecting, two states can be merged if they are indistinguishable, meaning there is no word that maps the pair to successor states with differing acceptance (one accepting, the other rejecting).

In the promise setting, because L_yes ∪ L_no does not cover the entire input space, some states are indefinite (neither accepting nor rejecting). We therefore ask whether there exists an accept/reject assignment for these indefinite states that maximizes the count of indistinguishable states within clusters while preserving the required distinguishability of the original accept/reject states specified by the promise problem.

In other words, we set up a Boolean satisfiability problem: assign a Boolean variable to each state (true = accepting, false = rejecting), add inequality constraints for the known accepting/rejecting states from the input DFA, then add equality constraints to enforce indistinguishability within each cluster, and test whether the constraints are satisfiable.

In [6]:
from z3 import Bool, Solver, sat
eq_clusters = [0, 1]
ineq_pair = (0, 3)

indices = list(range(num_states))
bool_vars = [Bool(f"x_{i}") for i in indices]

for k in eq_clusters:
    solver = Solver()

    # Add inequality constraint for the known accepting and rejecting states.
    solver.add(bool_vars[ineq_pair[0]] != bool_vars[ineq_pair[1]])

    # Add equality constraints for all pairs in the cluster.
    eqps = get_keys_from_value(cyclic_pair_classes, k)
    for i, j in eqps:
        solver.add(bool_vars[i] == bool_vars[j])

    if solver.check() == sat:
        model = solver.model()
        print(f"Cluster {k}: SAT with model: {model}")
    else:
        print(f"Cluster {k}: UNSAT")

Cluster 0: UNSAT
Cluster 1: SAT with model: [x_2 = False,
 x_4 = False,
 x_0 = False,
 x_3 = True,
 x_1 = True,
 x_5 = True]


The solver found out, that an alternating accept reject pattern for the 6 states is a solution, which indeed corresponds to the known optimal 2-state DFA for this promise problem! NExt, let us look at an example where the promise version of Hopcroft's algorithm can find a smaller DFA than the candidate DFA, which is the promise problem for the single-qubit Clifford group (TODO: ref to thesis).

In [7]:
num_states = 6
states = [DfaState(state_id=f'{i}') for i in range(num_states)]
input_alphabet = ['s','h']

states[0].is_accepting=True


states[0].transitions['s'] = states[1]
states[0].transitions['h'] = states[4]

states[1].transitions['s'] = states[2]
states[1].transitions['h'] = states[3]

states[2].transitions['s'] = states[3]
states[2].transitions['h'] = states[5]

states[3].transitions['s'] = states[0]
states[3].transitions['h'] = states[1]

states[4].transitions['s'] = states[4]
states[4].transitions['h'] = states[0]

states[5].transitions['s'] = states[5]
states[5].transitions['h'] = states[2]

sh_automaton = Dfa(states[0], states)

In [8]:
sh_pair_classes = {}

seed_pair = (0,1)
populate_pair_classes(seed_pair, sh_pair_classes, sh_automaton)

seed_pair = (0,4)
populate_pair_classes(seed_pair, sh_pair_classes, sh_automaton)

seed_pair = (2,5)
populate_pair_classes(seed_pair, sh_pair_classes, sh_automaton)

seed_pair = (0,2)
populate_pair_classes(seed_pair, sh_pair_classes, sh_automaton)

{(0, 1): 0,
 (1, 2): 0,
 (2, 3): 0,
 (3, 5): 0,
 (3, 4): 0,
 (1, 5): 0,
 (0, 4): 0,
 (1, 4): 0,
 (2, 4): 0,
 (0, 3): 0,
 (0, 5): 0,
 (2, 5): 0,
 (0, 2): 1,
 (1, 3): 1,
 (4, 5): 1}

In [9]:
check_pair_classification_complete(sh_pair_classes, num_states)

(True, [])

In [10]:
from z3 import Bool, Solver, sat
eq_clusters = [0]
ineq_pair = (0, 2)

indices = list(range(num_states))
bool_vars = [Bool(f"x_{i}") for i in indices]

for k in eq_clusters:
    solver = Solver()

    # Add inequality constraint for the known accepting and rejecting states.
    solver.add(bool_vars[ineq_pair[0]] != bool_vars[ineq_pair[1]])

    # Add equality constraints for all pairs in the cluster.
    eqps = get_keys_from_value(sh_pair_classes, k)
    for i, j in eqps:
        solver.add(bool_vars[i] == bool_vars[j])

    if solver.check() == sat:
        model = solver.model()
        print(f"Cluster {k}: SAT with model: {model}")
    else:
        print(f"Cluster {k}: UNSAT")

Cluster 0: UNSAT


We have only two cluster types. Cluster 1 is excluded from merging because its states are already distinguishable under the original accept/reject labeling. The SAT solver confirms that the states in cluster 0 cannot be labeled in a way that complies with the original accept/reject constraints. Therefore, the promise problem Cl is a strong candidate for quantum advantage: while it seems like a 6‑state DFA is required, we know that a single qubit QFA can solve it!